# 🛡️ SentinelFlow AI SOC (`sentinelflow-ai-soc`): End-to-End Engineering Notebook

**Author:** Ragib Nadim (Department of Computer Science, AIUB)  
**Environment:** Windows 11 Physical Workstation (`192.168.0.102`) + Ubuntu Server VM (`192.168.0.50`) on Bridged LAN  
**Core Stack:** Wazuh v4.14.10 (Manager + Indexer), NetAlertX (`ARPSCAN`), Grafana OSS (`v11+`), OpenSearch & Infinity JSONata Plugins  
**Future Work:** Airia AI Cloud Autonomous Alert Triage & Closed-Loop SOAR Integration

---

## 1. Executive Summary & Architectural Overview
**SentinelFlow AI SOC** unifies endpoint File Integrity Monitoring (FIM), continuous Layer-2 LAN asset discovery, and centralized SIEM observability into a single real-time Grafana dashboard (`SentinelFlow_AI_SOC`, UID: `adktrrf`). Unlike isolated home-lab setups where endpoint alerts and network discovery live in separate consoles, SentinelFlow bridges both telemetry streams onto a single operational view while reserving live data hooks (**Latest Airia risk** and **Latest alert name**) for Phase 3 **Airia AI** agentic triage.

![SentinelFlow Architecture](../images/wazuh_airia_grafana_workflow_labeled.png)

## 2. Phase 1: Wazuh SIEM Deployment & Real-Time File Integrity Monitoring (FIM)

### 2.1 Endpoint Enrollment & Baseline Telemetry
1. Deployed **Wazuh Manager & OpenSearch Indexer v4.14.10** on Ubuntu Server (`node01`, `192.168.0.50`).
2. Enrolled the physical Windows 11 host (`DESKTOP-HOQHBSK`, `192.168.0.102`, Agent ID `001`) into the `default` agent group.
3. Verified baseline Security Configuration Assessment (CIS Microsoft Windows 11 Enterprise Benchmark v3.0.0) and agent heartbeat status.

| Wazuh Overview | Enrolled Endpoint (`DESKTOP-HOQHBSK`) |
| :---: | :---: |
| ![Wazuh Overview](../images/01-Wazuh%20Dashboard.png) | ![Wazuh Endpoints](../images/02-Wazuh%20Dashboard.png) |

### 2.2 Configuring Real-Time Syscheck (`ossec.conf`)
To detect unauthorized file creation, tampering, or deletion immediately rather than waiting for scheduled 12-hour scans, `<syscheck>` on `DESKTOP-HOQHBSK` was configured with `realtime="yes"`, `check_all="yes"`, and `report_changes="yes"` watching `E:\Personal Projects\sentinelflow-ai-soc\monitored_directory`:

```xml
<synchronization>
  <enabled>yes</enabled>
  <interval>5m</interval>
  <max_eps>10</max_eps>
</synchronization>
<directories check_all="yes" report_changes="yes" realtime="yes">E:\Personal Projects\sentinelflow-ai-soc\monitored_directory</directories>
```

| Modifying `ossec.conf` | PowerShell Adversarial FIM Lifecycle Test |
| :---: | :---: |
| ![Modifying ossec.conf](../images/04-Modifying%20ossec.conf.png) | ![FIM Test](../images/05-File%20Integrity%20Monitoring%20Test.png) |

### 2.3 Validation Results (Before vs. After Event Capture)
Running the PowerShell lifecycle test (`Set-Content` -> `Add-Content` -> `Remove-Item` with 7-second intervals) generated **18 real-time FIM events** across Wazuh Rules `554` (File added, Level 5), `550` (Integrity checksum changed, Level 7), and `553` (File deleted, Level 7):

| Before Event Capture | After Event Capture (18 Hits) |
| :---: | :---: |
| ![Before Events](../images/06-Before%20Events%20Capture.png) | ![After Events](../images/07-After%20Events%20Capture.png) |

## 3. Phase 2: NetAlertX LAN Discovery & Unified Grafana Integration

### 3.1 Continuous ARP Scanning with NetAlertX
Because endpoints can suffer attacks or rogue hardware insertions outside Wazuh's visibility, **NetAlertX** was deployed on `192.168.0.50:20212` using a bridged network interface:
- Configured `ARPSCAN` on a 5-minute cron schedule (`*/5 * * * *`) with `ARPSCAN_SET_ALWAYS` set to `devMac` and `devLastIP`.
- Discovered **9 active subnet devices** (`Ubuntu` at `192.168.0.50`, `_gateway` at `192.168.0.1`, and LAN clients `192.168.0.100–.105`).

| NetAlertX Dashboard (9 Devices) | ARPSCAN Scheduler Settings |
| :---: | :---: |
| ![NetAlertX Dashboard](../images/13-NetAlertX-Devices%20Discovery.png) | ![ARPSCAN Config](../images/12-NetAlertX-ARP%20Scan.png) |

### 3.2 Grafana Datasource Provisioning & Troubleshooting Log
Two datasources were connected to Grafana (`http://192.168.0.50:3000`):
1. **Wazuh Indexer (`SentinelFlow-Wazuh-Indexer-2`, UID `fg5ao8lrrde68e`):** Connected via the **OpenSearch** plugin to `https://192.168.0.50:9200` (`wazuh-alerts-*`, `@timestamp`).
2. **NetAlertX REST API (`NetAlertX Infinity`, UID `cg5b0m1omr6kgb`):** Connected via the **Infinity** plugin to `http://192.168.0.50:20212/devices`.

#### Engineering Fix: Resolving `No data` & `[[null]]` Frames in Infinity Stat Panels
- **Root Cause:** Querying `/devices` with a strict JSONata filter (`devices[$lowercase($string(devStatus)) in ["online", "on-line"]]`) returned empty data frames (`values: [[null]]`) when `devStatus` was unset or formatted differently in NetAlertX's API payload.
- **Resolution:** Set `Rows/Root` to `devices` with column selector `devName` (`Online Devices`), removed conflicting `Group by` transformations lacking group keys, and configured Grafana's native **Value options -> Show: Calculate -> Calculation: Count**, cleanly rendering **`9` Online devices** alongside **793–796 FIM events**.

| OpenSearch Health Check | Final Unified `SentinelFlow_AI_SOC` Dashboard |
| :---: | :---: |
| ![OpenSearch Connection](../images/16-Connection-Wazuh-to-Grafana.png) | ![Final Dashboard](../images/19-Dashboard%20Updates.png) |

In [ ]:
# Verification Script: Inspect SentinelFlow_AI_SOC Dashboard JSON Structure
import json
from pathlib import Path

dash_path = Path("../grafana/dashboards/SentinelFlow_AI_SOC.json")
dashboard = json.loads(dash_path.read_text(encoding="utf-8"))

print(f"Dashboard Title : {dashboard['title']} (UID: {dashboard['uid']})")
print(f"Refresh Rate    : {dashboard['refresh']} | Time Window: {dashboard['time']['from']} -> {dashboard['time']['to']}")
print("\nConfigured Panels:")
for p in dashboard["panels"]:
    if p["type"] != "row":
        ds_uid = p.get("datasource", {}).get("uid", "N/A")
        print(f"  - [ID {p['id']:02d}] {p['title']:<24} | Type: {p['type']:<10} | Datasource UID: {ds_uid}")

## 4. Phase 3 (Future Work): Airia AI Autonomous Alert Triage & SOAR Pipeline

Notice the two reserved panels in the top **Overview** row of `SentinelFlow_AI_SOC`: **Latest Airia risk** and **Latest alert name**. These panels are pre-wired to receive real-time LLM triage verdicts from **Airia AI**.

### 4.1 Planned Airia AI Architecture
1. **Wazuh Custom Integration (`wazuh_airia_webhook.py`):** When Wazuh Manager triggers an alert with `rule.level >= 7` (such as FIM checksum tampering `Rule 550` or deletion `Rule 553`), `wazuh-integratord` invokes the Python bridge.
2. **Cross-Layer Asset Enrichment:** The bridge queries NetAlertX (`http://192.168.0.50:20212/devices`) to attach MAC address, device type, and asset trust status (`known_asset: true/false`) to the alert.
3. **Multi-Agent Airia AI Pipeline:**
   - **Triage & MITRE Mapper Agent:** Classifies intent (e.g., benign developer edit vs. ransomware canary modification `T1486` / timestomping `T1070.006`).
   - **Risk Scoring Agent:** Outputs an integer risk score (`0–100`) and concise incident headline.
   - **Containment Advisor Agent:** Recommends Wazuh Active Response actions (`firewall-drop` or process kill).
4. **Closed-Loop Dashboard Update:** Airia AI writes the structured JSON verdict into OpenSearch (`airia-triage-alerts`), automatically lighting up **Latest Airia risk** and **Latest alert name** on Grafana.

### 4.2 Prototype Simulation of Airia AI Triage Payload
Run the cell below to preview the exact enriched JSON payload that Phase 3 sends to Airia AI and indexes back into OpenSearch.

In [ ]:
# Prototype Simulation: Airia AI Triage Verdict Generator
import json
from datetime import datetime, timezone

sample_wazuh_fim_alert = {
    "id": "1732513111.849201",
    "rule": {"id": "550", "level": 7, "description": "Integrity checksum changed."},
    "agent": {"id": "001", "name": "DESKTOP-HOQHBSK", "ip": "192.168.0.102"},
    "syscheck": {
        "path": "e:\\personal projects\\sentinelflow-ai-soc\\monitored_directory\\test.txt",
        "event": "modified",
        "sha256_after": "3608bca1e44ea6c4d268eb6db02260269892c0b42b86bbf1e77a6fa16c3c9282"
    }
}

simulated_airia_verdict = {
    "@timestamp": datetime.now(timezone.utc).isoformat(),
    "source_alert_id": sample_wazuh_fim_alert["id"],
    "agent_name": sample_wazuh_fim_alert["agent"]["name"],
    "agent_ip": sample_wazuh_fim_alert["agent"]["ip"],
    "airia_risk_score": 68,
    "airia_alert_title": "Suspicious File Checksum Modification in Monitored Directory (T1565.001)",
    "severity_label": "MEDIUM",
    "mitre_techniques": ["T1565.001", "T1070.004"],
    "network_context": {"known_asset": True, "mac": "ea:a8:51:f8:8a:2d", "scanner": "ARPSCAN"},
    "remediation_steps": [
        "Verify process handle accessing E:\\Personal Projects\\sentinelflow-ai-soc\\monitored_directory",
        "Compare SHA-256 hash against authorized build artifacts",
        "Enable Wazuh Active Response quarantine if unauthorized encryption behavior persists"
    ]
}

print(json.dumps(simulated_airia_verdict, indent=2))